# Vault — Source Data Profiling

This notebook performs the initial quality and structure assessment of the raw Berka banking datasets used in the Vault project.

The goal is to understand each source before analytical modeling or feature engineering. Profiling covers table structure, missingness, duplicates, primary-key integrity, categorical distributions, numeric summaries, and date coverage.

> **Scope:** This notebook focuses on checks within each individual source table. Foreign-key integrity and cross-table relationship/cardinality checks are handled separately after single-table profiling.


In [1]:
import pandas as pd

### Source Profiling Checklist — repeat for each CSV
- Load the file and inspect the first rows
- Check number of rows and columns
- Check column names
- Check inferred data types
- Check missing values per column, both count and %
- Check fully duplicated rows
- Check primary-key uniqueness and nulls
- Check unique values / cardinality per column
- Inspect categorical value counts
- Check numeric columns: min, max, mean, median and suspicious values
- Check date columns: parsing validity, minimum date and maximum date
- Check foreign keys against their parent tables for orphan records
- Check expected relationships/cardinality, e.g. one-to-one, one-to-many, many-to-many
- Record any anomalies, assumptions, or data-quality concerns without fixing them yet

## 1. Reusable Source Profiling Function

The function below standardizes checks repeated across source tables while allowing table-specific primary keys and date columns to be supplied as parameters.

`client.csv` requires special handling because `birth_number` encodes the birth date and gender. Other source dates use the dataset's `YYMMDD` representation; where a trailing midnight timestamp is present, only the date component is relevant for this profiling step.


In [215]:
def source_profiling(
    path,
    csv,
    primary_key=None,
    date_columns=["birth_number"]
):
    # Load file
    df = pd.read_csv(f"{path}/{csv}.csv", sep=";")

    # Shape
    print("\n--- Structure ---")
    print(f"Shape: {df.shape}")

    # Column names
    column_names = df.columns.to_list()
    print(f"Columns: {column_names}")

    # Inferred data types
    print(f"\nData types:\n{df.dtypes}")

    # Missing values
    print("\n--- Data Quality ---")
    missing_values = df.isna().sum().to_frame(name="missing_count")
    missing_values["missing_pct"] = (
        missing_values["missing_count"] / df.shape[0] * 100
    )
    print(f"\nMissing values:\n{missing_values}")

    # Fully duplicated rows
    fully_dup_rows = df.duplicated().sum()
    print(f"\nFully duplicated rows: {fully_dup_rows}")

    # Primary key uniqueness and nulls
    if primary_key:
        pk_duplicates = df[primary_key].duplicated().sum()
        pk_nulls = df[primary_key].isna().sum()

        print(f"\nPrimary key duplicates: {pk_duplicates}")
        print(f"Primary key nulls: {pk_nulls}")
    else:
        print("\nTable has no primary key")

    # Categorical columns
    categorical_columns = df.select_dtypes(
        include=["category", "object", "bool"]
    ).columns.tolist()

    print("\n--- Categorical Columns ---")

    for categorical_col in categorical_columns:
        print(f"\nColumn {categorical_col} values:")
        print(df[categorical_col].value_counts(dropna=False))

    # Numeric columns
    numeric_columns = df.select_dtypes(include="number").columns.tolist()

    print("\n--- Numeric Summary ---")
    print(df[numeric_columns].describe())

    # Date columns
    if date_columns:
        for date_col in date_columns:

            # Special case: client.csv birth_number
            if csv == "client":

                def parse_birth_date(code):
                    birth_number = int(code)

                    # Decode female month
                    month = (birth_number // 100) % 100

                    if month > 50:
                        birth_number -= 5000

                    # Extract date components
                    year = 1900 + (birth_number // 10000)
                    month = (birth_number // 100) % 100
                    day = birth_number % 100

                    return pd.Timestamp(
                        year=year,
                        month=month,
                        day=day
                    )

                df[date_col] = df[date_col].apply(parse_birth_date)

            # All other YYMMDD date columns. For card.csv, the trailing time is 00:00:00,
            # so only the first six characters are needed for date profiling.
            else:
                df[date_col] = pd.to_datetime(
                    df[date_col].astype(str).str[0:6],
                    format="%y%m%d"
                )

            # Date coverage
            min_date = df[date_col].min()
            max_date = df[date_col].max()

            print(
                f"\n{date_col} date coverage: "
                f"{min_date} to {max_date}"
            )

    # Final dataframe information
    print("\n--- Final DataFrame Info ---")
    df.info()

    return df

## 2. Dataset Configuration

Each source table is mapped to its expected primary key and date column(s). Keeping this metadata outside the profiling function makes the function reusable without hard-coding most table-specific configuration.


In [217]:
file_path = "../data/raw"

datasets = {
    "account": {
        "pk": "account_id",
        "date_col": ["date"]
    },
    "card": {
        "pk": "card_id",
        "date_col": ["issued"]
    },
    "client": {
        "pk": "client_id",
        "date_col": ["birth_number"]
    },
    "disp": {
        "pk": "disp_id",
        "date_col": []
    },
    "district": {
        "pk": "A1",
        "date_col": []
    },
    "loan": {
        "pk": "loan_id",
        "date_col": ["date"]
    },
    "order": {
        "pk": "order_id",
        "date_col": []
    },
    "trans": {
        "pk": "trans_id",
        "date_col": ["date"]
    }
}

## 3. Run Source Profiling

The profiling function is executed for every raw source table using the configuration defined above. Review the outputs for structural issues, unexpected missingness, invalid primary keys, unusual category values, suspicious numeric ranges, and date coverage anomalies.


In [219]:
for dataset, config in datasets.items():
    print(f"\n{'=' * 60}\n{dataset.upper()} DATASET PROFILING\n{'=' * 60}")

    source_profiling(
        file_path,
        csv=dataset,
        primary_key=config["pk"],
        date_columns=config["date_col"]
    )
    print("\n----------------------")


account dataset profiling
Shape: (4500, 4)

Columns:
['account_id', 'district_id', 'frequency', 'date']

Data types:
account_id      int64
district_id     int64
frequency      object
date            int64
dtype: object

Missing values:
             missing_count  missing_pct
account_id               0          0.0
district_id              0          0.0
frequency                0          0.0
date                     0          0.0

Fully duplicated rows: 0

Primary key duplicates: 0
Primary key nulls: 0

Categorical columns value counts:

Column frequency values:
frequency
POPLATEK MESICNE      4167
POPLATEK TYDNE         240
POPLATEK PO OBRATU      93
Name: count, dtype: int64

Numeric columns summary:
         account_id  district_id           date
count   4500.000000  4500.000000    4500.000000
mean    2786.067556    37.310444  951654.608667
std     2313.811984    25.177217   14842.188377
min        1.000000     1.000000  930101.000000
25%     1182.750000    13.000000  931227.0000

C:\Users\Nacho\AppData\Local\Temp\ipykernel_23288\1798666291.py:8: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{path}/{csv}.csv", sep=";")


Shape: (1056320, 10)

Columns:
['trans_id', 'account_id', 'date', 'type', 'operation', 'amount', 'balance', 'k_symbol', 'bank', 'account']

Data types:
trans_id        int64
account_id      int64
date            int64
type           object
operation      object
amount        float64
balance       float64
k_symbol       object
bank           object
account       float64
dtype: object

Missing values:
            missing_count  missing_pct
trans_id                0     0.000000
account_id              0     0.000000
date                    0     0.000000
type                    0     0.000000
operation          183114    17.335088
amount                  0     0.000000
balance                 0     0.000000
k_symbol           481881    45.618847
bank               782812    74.107467
account            760931    72.036031

Fully duplicated rows: 0

Primary key duplicates: 0
Primary key nulls: 0

Categorical columns value counts:

Column type values:
type
VYDAJ     634571
PRIJEM    405083

## 4. Next Step: Cross-Table Integrity

After reviewing the individual table profiles, validate relationships between sources. This includes checking foreign keys for orphan records and confirming expected relationship cardinalities such as one-to-many and many-to-many relationships.

Material anomalies or assumptions discovered during profiling should be recorded before moving into data preparation and analytical modeling.
